In [1]:
!pip install fastapi uvicorn sqlalchemy pydantic requests

In [2]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.orm import declarative_base, sessionmaker, Session
from fastapi.responses import JSONResponse

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
DATABASE_URL = "sqlite:///./employees.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()

print("Database configuration created successfully!")

Database configuration created successfully!


In [4]:
class Employee(Base):
    __tablename__ = "employees"

    id = Column(
        Integer,
        primary_key=True,
        index=True
    )

    name = Column(
        String,
        nullable=False
    )

    email = Column(
        String,
        unique=True,
        nullable=False
    )

    age = Column(
        Integer,
        nullable=False
    )

    department = Column(
        String,
        nullable=False
    )

    salary = Column(
        Float,
        nullable=False
    )


Base.metadata.create_all(bind=engine)

print("Employee table created successfully!")

Employee table created successfully!


In [5]:
app = FastAPI(
    title="Employee Management API",
    description="RESTful API for managing employee records",
    version="1.0.0"
)

print("FastAPI application created successfully!")

FastAPI application created successfully!


In [7]:
from pydantic import BaseModel, Field, ConfigDict


class EmployeeCreate(BaseModel):

    name: str = Field(
        ...,
        min_length=2,
        max_length=50,
        description="Employee name"
    )

    email: str = Field(
        ...,
        min_length=5,
        max_length=100,
        description="Employee email"
    )

    age: int = Field(
        ...,
        ge=18,
        le=65,
        description="Employee age"
    )

    department: str = Field(
        ...,
        min_length=2,
        max_length=50,
        description="Employee department"
    )

    salary: float = Field(
        ...,
        gt=0,
        description="Employee salary"
    )


class EmployeeResponse(BaseModel):

    id: int
    name: str
    email: str
    age: int
    department: str
    salary: float

    model_config = ConfigDict(
        from_attributes=True
    )


print("Data validation models created successfully!")

Data validation models created successfully!


In [8]:
def get_db():

    db = SessionLocal()

    try:
        yield db

    finally:
        db.close()


print("Database session function created!")

Database session function created!


In [9]:
@app.post(
    "/employees",
    response_model=EmployeeResponse,
    status_code=201,
    tags=["Employees"]
)
def create_employee(employee: EmployeeCreate):

    db = SessionLocal()

    try:

        # Check duplicate email
        existing_employee = (
            db.query(Employee)
            .filter(
                Employee.email == employee.email
            )
            .first()
        )

        if existing_employee:

            raise HTTPException(
                status_code=400,
                detail="Employee with this email already exists"
            )

        new_employee = Employee(
            name=employee.name,
            email=employee.email,
            age=employee.age,
            department=employee.department,
            salary=employee.salary
        )

        db.add(new_employee)

        db.commit()

        db.refresh(new_employee)

        return new_employee

    finally:

        db.close()

In [10]:
@app.get(
    "/employees",
    response_model=list[EmployeeResponse],
    tags=["Employees"]
)
def get_employees():

    db = SessionLocal()

    try:

        employees = db.query(Employee).all()

        return employees

    finally:

        db.close()

In [11]:
@app.get(
    "/employees/{employee_id}",
    response_model=EmployeeResponse,
    tags=["Employees"]
)
def get_employee(employee_id: int):

    db = SessionLocal()

    try:

        employee = (
            db.query(Employee)
            .filter(
                Employee.id == employee_id
            )
            .first()
        )

        if employee is None:

            raise HTTPException(
                status_code=404,
                detail="Employee not found"
            )

        return employee

    finally:

        db.close()

In [12]:
@app.put(
    "/employees/{employee_id}",
    response_model=EmployeeResponse,
    tags=["Employees"]
)
def update_employee(
    employee_id: int,
    employee_data: EmployeeCreate
):

    db = SessionLocal()

    try:

        employee = (
            db.query(Employee)
            .filter(
                Employee.id == employee_id
            )
            .first()
        )

        if employee is None:

            raise HTTPException(
                status_code=404,
                detail="Employee not found"
            )

        # Check whether email belongs to another employee
        existing_email = (
            db.query(Employee)
            .filter(
                Employee.email == employee_data.email,
                Employee.id != employee_id
            )
            .first()
        )

        if existing_email:

            raise HTTPException(
                status_code=400,
                detail="Another employee already uses this email"
            )

        employee.name = employee_data.name
        employee.email = employee_data.email
        employee.age = employee_data.age
        employee.department = employee_data.department
        employee.salary = employee_data.salary

        db.commit()

        db.refresh(employee)

        return employee

    finally:

        db.close()

In [13]:
@app.delete(
    "/employees/{employee_id}",
    tags=["Employees"]
)
def delete_employee(employee_id: int):

    db = SessionLocal()

    try:

        employee = (
            db.query(Employee)
            .filter(
                Employee.id == employee_id
            )
            .first()
        )

        if employee is None:

            raise HTTPException(
                status_code=404,
                detail="Employee not found"
            )

        db.delete(employee)

        db.commit()

        return {
            "message": "Employee deleted successfully",
            "employee_id": employee_id
        }

    finally:

        db.close()

In [14]:
@app.exception_handler(Exception)
async def global_exception_handler(request, exc):

    return JSONResponse(
        status_code=500,
        content={
            "error": "Internal Server Error",
            "message": "Something went wrong on the server"
        }
    )


print("Error handling configured successfully!")

Error handling configured successfully!


In [15]:
@app.get(
    "/",
    tags=["Health"]
)
def home():

    return {
        "message": "Employee Management API is running",
        "status": "success",
        "documentation": "/docs"
    }

In [16]:
import inspect

source_code = """
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.orm import declarative_base, sessionmaker
from fastapi.responses import JSONResponse

DATABASE_URL = "sqlite:///./employees.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()


class Employee(Base):
    __tablename__ = "employees"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    email = Column(String, unique=True, nullable=False)
    age = Column(Integer, nullable=False)
    department = Column(String, nullable=False)
    salary = Column(Float, nullable=False)


Base.metadata.create_all(bind=engine)


app = FastAPI(
    title="Employee Management API",
    description="RESTful API for managing employee records",
    version="1.0.0"
)


class EmployeeCreate(BaseModel):

    name: str = Field(..., min_length=2, max_length=50)
    email: str = Field(..., min_length=5, max_length=100)
    age: int = Field(..., ge=18, le=65)
    department: str = Field(..., min_length=2, max_length=50)
    salary: float = Field(..., gt=0)


class EmployeeResponse(BaseModel):

    id: int
    name: str
    email: str
    age: int
    department: str
    salary: float

    class Config:
        from_attributes = True


@app.get("/", tags=["Health"])
def home():

    return {
        "message": "Employee Management API is running",
        "status": "success",
        "documentation": "/docs"
    }


@app.post(
    "/employees",
    response_model=EmployeeResponse,
    status_code=201,
    tags=["Employees"]
)
def create_employee(employee: EmployeeCreate):

    db = SessionLocal()

    try:

        existing_employee = (
            db.query(Employee)
            .filter(Employee.email == employee.email)
            .first()
        )

        if existing_employee:

            raise HTTPException(
                status_code=400,
                detail="Employee with this email already exists"
            )

        new_employee = Employee(
            name=employee.name,
            email=employee.email,
            age=employee.age,
            department=employee.department,
            salary=employee.salary
        )

        db.add(new_employee)
        db.commit()
        db.refresh(new_employee)

        return new_employee

    finally:

        db.close()


@app.get(
    "/employees",
    response_model=list[EmployeeResponse],
    tags=["Employees"]
)
def get_employees():

    db = SessionLocal()

    try:

        return db.query(Employee).all()

    finally:

        db.close()


@app.get(
    "/employees/{employee_id}",
    response_model=EmployeeResponse,
    tags=["Employees"]
)
def get_employee(employee_id: int):

    db = SessionLocal()

    try:

        employee = (
            db.query(Employee)
            .filter(Employee.id == employee_id)
            .first()
        )

        if employee is None:

            raise HTTPException(
                status_code=404,
                detail="Employee not found"
            )

        return employee

    finally:

        db.close()


@app.put(
    "/employees/{employee_id}",
    response_model=EmployeeResponse,
    tags=["Employees"]
)
def update_employee(
    employee_id: int,
    employee_data: EmployeeCreate
):

    db = SessionLocal()

    try:

        employee = (
            db.query(Employee)
            .filter(Employee.id == employee_id)
            .first()
        )

        if employee is None:

            raise HTTPException(
                status_code=404,
                detail="Employee not found"
            )

        existing_email = (
            db.query(Employee)
            .filter(
                Employee.email == employee_data.email,
                Employee.id != employee_id
            )
            .first()
        )

        if existing_email:

            raise HTTPException(
                status_code=400,
                detail="Another employee already uses this email"
            )

        employee.name = employee_data.name
        employee.email = employee_data.email
        employee.age = employee_data.age
        employee.department = employee_data.department
        employee.salary = employee_data.salary

        db.commit()
        db.refresh(employee)

        return employee

    finally:

        db.close()


@app.delete(
    "/employees/{employee_id}",
    tags=["Employees"]
)
def delete_employee(employee_id: int):

    db = SessionLocal()

    try:

        employee = (
            db.query(Employee)
            .filter(Employee.id == employee_id)
            .first()
        )

        if employee is None:

            raise HTTPException(
                status_code=404,
                detail="Employee not found"
            )

        db.delete(employee)
        db.commit()

        return {
            "message": "Employee deleted successfully",
            "employee_id": employee_id
        }

    finally:

        db.close()


@app.exception_handler(Exception)
async def global_exception_handler(request, exc):

    return JSONResponse(
        status_code=500,
        content={
            "error": "Internal Server Error",
            "message": "Something went wrong on the server"
        }
    )
"""

with open("main.py", "w", encoding="utf-8") as file:
    file.write(source_code)

print("main.py created successfully!")

main.py created successfully!


In [17]:
requirements = """fastapi
uvicorn
sqlalchemy
pydantic
requests
"""

with open(
    "requirements.txt",
    "w"
) as file:

    file.write(requirements)

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [18]:
import subprocess
import sys
import time

server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ]
)

time.sleep(3)

print("FastAPI server started!")
print("API: http://127.0.0.1:8000")
print("Documentation: http://127.0.0.1:8000/docs")

FastAPI server started!
API: http://127.0.0.1:8000
Documentation: http://127.0.0.1:8000/docs


In [19]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/"
)

print("Status code:", response.status_code)
print("Response:")

print(response.json())

Status code: 200
Response:
{'message': 'Employee Management API is running', 'status': 'success', 'documentation': '/docs'}


In [20]:
employee_data = {
    "name": "Rahul",
    "email": "rahul@example.com",
    "age": 22,
    "department": "IT",
    "salary": 45000
}

response = requests.post(
    "http://127.0.0.1:8000/employees",
    json=employee_data
)

print("Status code:", response.status_code)
print(response.json())

Status code: 201
{'id': 1, 'name': 'Rahul', 'email': 'rahul@example.com', 'age': 22, 'department': 'IT', 'salary': 45000.0}


In [21]:
response = requests.get(
    "http://127.0.0.1:8000/employees"
)

print("Status code:", response.status_code)

print(
    response.json()
)

Status code: 200
[{'id': 1, 'name': 'Rahul', 'email': 'rahul@example.com', 'age': 22, 'department': 'IT', 'salary': 45000.0}]


In [22]:
response = requests.get(
    "http://127.0.0.1:8000/employees/1"
)

print("Status code:", response.status_code)

print(
    response.json()
)

Status code: 200
{'id': 1, 'name': 'Rahul', 'email': 'rahul@example.com', 'age': 22, 'department': 'IT', 'salary': 45000.0}


In [23]:
updated_employee = {
    "name": "Rahul Kumar",
    "email": "rahul@example.com",
    "age": 23,
    "department": "Data Science",
    "salary": 55000
}

response = requests.put(
    "http://127.0.0.1:8000/employees/1",
    json=updated_employee
)

print("Status code:", response.status_code)

print(
    response.json()
)

Status code: 200
{'id': 1, 'name': 'Rahul Kumar', 'email': 'rahul@example.com', 'age': 23, 'department': 'Data Science', 'salary': 55000.0}


In [24]:
response = requests.delete(
    "http://127.0.0.1:8000/employees/1"
)

print("Status code:", response.status_code)

print(
    response.json()
)

Status code: 200
{'message': 'Employee deleted successfully', 'employee_id': 1}


In [25]:
invalid_employee = {
    "name": "A",
    "email": "wrong",
    "age": 10,
    "department": "IT",
    "salary": -5000
}

response = requests.post(
    "http://127.0.0.1:8000/employees",
    json=invalid_employee
)

print("Status code:", response.status_code)

print(
    response.json()
)

Status code: 422
{'detail': [{'type': 'string_too_short', 'loc': ['body', 'name'], 'msg': 'String should have at least 2 characters', 'input': 'A', 'ctx': {'min_length': 2}}, {'type': 'greater_than_equal', 'loc': ['body', 'age'], 'msg': 'Input should be greater than or equal to 18', 'input': 10, 'ctx': {'ge': 18}}, {'type': 'greater_than', 'loc': ['body', 'salary'], 'msg': 'Input should be greater than 0', 'input': -5000, 'ctx': {'gt': 0.0}}]}


In [26]:
response = requests.get(
    "http://127.0.0.1:8000/employees/99999"
)

print("Status code:", response.status_code)

print(
    response.json()
)

Status code: 404
{'detail': 'Employee not found'}


In [27]:
import webbrowser

webbrowser.open(
    "http://127.0.0.1:8000/docs"
)

True

In [32]:
readme = '''
# REST API Development - Employee Management API

## Overview

This project is a RESTful API developed using FastAPI and SQLite.

The API provides CRUD operations for managing employee records.

## Technologies Used

- Python
- FastAPI
- SQLite
- SQLAlchemy
- Pydantic
- Uvicorn
- Requests
- Jupyter Notebook

## Features

- RESTful API
- CRUD operations
- SQLite database integration
- Input validation
- Error handling
- JSON responses
- Swagger API documentation
- ReDoc API documentation

## API Endpoints

| Method | Endpoint | Description |
|--------|----------|-------------|
| GET | / | API health check |
| POST | /employees | Create employee |
| GET | /employees | Get all employees |
| GET | /employees/{id} | Get employee by ID |
| PUT | /employees/{id} | Update employee |
| DELETE | /employees/{id} | Delete employee |

## Employee Data Model

| Field | Type | Description |
|-------|------|-------------|
| id | Integer | Unique employee ID |
| name | String | Employee name |
| email | String | Employee email |
| age | Integer | Employee age |
| department | String | Employee department |
| salary | Float | Employee salary |

## Validation

The API validates:

- Required fields
- Name length
- Email length
- Age between 18 and 65
- Positive salary
- Duplicate email addresses

## Error Handling

The API handles:

- Employee not found
- Duplicate email
- Invalid input
- Invalid data types
- Server errors

## Installation

Install the required packages:

    pip install -r requirements.txt

## Run the API

Run:

    uvicorn main:app --reload

The API will be available at:

    http://127.0.0.1:8000

## API Documentation

Swagger UI:

    http://127.0.0.1:8000/docs

ReDoc:

    http://127.0.0.1:8000/redoc

## Example POST Request

    {
        "name": "Rahul",
        "email": "rahul@example.com",
        "age": 22,
        "department": "IT",
        "salary": 45000
    }

## Example Response

    {
        "id": 1,
        "name": "Rahul",
        "email": "rahul@example.com",
        "age": 22,
        "department": "IT",
        "salary": 45000
    }

## Database

This project uses SQLite.

Database file:

    employees.db

## Author

Internship Task 3 - REST API Development
'''

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme)

print("README.md created successfully!")

README.md created successfully!


In [33]:
requirements = """fastapi
uvicorn
sqlalchemy
pydantic
requests
"""

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements)

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [34]:
import os

files = [
    "main.py",
    "README.md",
    "requirements.txt",
    "employees.db"
]

print("====================================")
print("       PROJECT FILE CHECK")
print("====================================")

for file in files:
    if os.path.exists(file):
        print("✓", file)
    else:
        print("✗", file, "NOT FOUND")

       PROJECT FILE CHECK
✓ main.py
✓ README.md
✓ requirements.txt
✓ employees.db
